# Stage 2: Advanced Embedding Models Training and Analysis

## Libraries

In [39]:
!pip install gensim plotly scikit-learn
!pip install -q -U transformers peft bitsandbytes trl accelerate datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.6 MB/s eta 0:00:00


In [40]:
import os
import re
import multiprocessing
from collections import Counter
import pandas as pd
import numpy as np
import ast
from gensim.models import Word2Vec, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
import plotly.express as px
import plotly.io as pio
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments
from trl import SFTTrainer

ValueError: pyarrow.lib.IpcReadOptions size changed, may indicate binary incompatibility. Expected 112 from C header, got 104 from PyObject

In [44]:
from peft import LoraConfig, get_peft_model

## Environment setup

In your Google Drive, create a folder "CLT" and upload the csv ai_media_norm.csv

Switch: "local" (VS Code + GitHub) | "colab" (Google Colab + Drive)

In [24]:
ENV = "colab"
pio.renderers.default = "colab" if ENV == "colab" else "notebook"

In [25]:
if ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 1 Dataset Preparation and Export

*Lead: Alla - Review: Marisa*

At this point, the Stage 1 preprocessing pipeline is complete [(Link to colab)](https://colab.research.google.com/github/allarom/clt-26/blob/main/clt.ipynb).
The dataset of Stage 1 is about 1.5 GB and we decided to
save only necessary data to `ai_media_norm.csv` containing the 8 columns needed for Stage 2, discarding all intermediate token and lemma variants that were used internally during cleaning.
This avoids repeating the computationally expensive and long spaCy processing (tokenization, lemmatization).

**Saved columns**
- `date`
- `domain`
- `url`
- `tags`
- `title`
- `content`
- `title_lemma_norm`
- `content_lemma_norm`.

This keeps the file small and focused — Stage 2 only needs the original metadata, the cleaned readable text, and the final normalized lemma lists for embedding training.

The following code show what was done at the end of the preprocessing step in Stage 1. We included it merely for explanatory purposes, the code does not need to be run in this stage.



In [26]:
# This code doesnt need to be run, its just a demonstration of how we saved the normalized data
# in Stage 1 file https://colab.research.google.com/github/allarom/clt-26/blob/main/clt.ipynb.

# Save only the columns needed for Stage 2 as a checkpoint
# stage2_input_cols = [
#     'date', 'domain', 'url', 'tags',
#     'title', 'content',
#     'title_lemma_norm', 'content_lemma_norm',
# ]

# if ENV == "colab":
#     norm_path = '/content/drive/My Drive/CLT/ai_media_norm.csv'
# else:
#     norm_path = 'data/ai_media_norm.csv'

# os.makedirs(os.path.dirname(norm_path), exist_ok=True)
# df_norm[stage2_input_cols].to_csv(norm_path, index=False, encoding='utf-8')

# print("Saved to:", norm_path)
# print("Shape:", df_norm[stage2_input_cols].shape)

## 1.1 Load Stage 1 Checkpoint


We load the preprocessed dataset (`ai_media_norm.csv`).

Since CSV does not preserve Python list types, we use `ast.literal_eval` to restore `title_lemma_norm`, `content_lemma_norm`, and `tags` back to proper Python lists. The `date` column is also re-parsed to `datetime`.

In [27]:
# Load the Stage 1 checkpoint
if ENV == "colab":
    norm_path = '/content/drive/My Drive/CLT/ai_media_norm.csv'
else:
    norm_path = 'data/ai_media_norm.csv'

df_norm_loaded = pd.read_csv(norm_path, encoding='utf-8')

# Restore list columns serialized as strings by CSV
for col in ['title_lemma_norm', 'content_lemma_norm', 'tags']:
    df_norm_loaded[col] = df_norm_loaded[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

df_norm_loaded['date'] = pd.to_datetime(df_norm_loaded['date'])

print("Loaded shape:", df_norm_loaded.shape)


Loaded shape: (16518, 8)


## 1.2 Build Stage 2 Columns

For further analysis, we create the following columns:
- **`doc_id`**: Stable zero-padded identifier (e.g. `doc_00042`) — survives CSV export and links model outputs back to source articles.
- **`text_clean_light`**: Readable text from cleaned `title` + `content` — used for display, search, and labeling. Stop words kept, natural sentence structure preserved.
- **`text_for_embedding`**: Input for Word2Vec  — `title_lemma_norm` + `content_lemma_norm` joined as a space-separated string. Stop words removed, noise filtered.
- **`token_count`**: Number of tokens in `text_for_embedding`, used for quality filtering in the next step.
- **`split`**: Reproducible 80/20 train/val assignment with fixed random seed (42).

🟨🟨🟨🟨 Marisa: Ich verstehe bei jedem einzelnen Punkt nicht, warum das jeweils notwendig ist. Kannst du das noch ausführen, warum das benötigt wird (z.B. Voraussetzung für embedding, weil es nur die Form "xyz" versteht. Beim ersten Punkt: Was geht konkret beim PDF Export verloren? --> Die Notebooks sollen ja auch wie eine Art Tutorial aufgebaut sein, deshalb wäre es eh gut, wenn es hier erklärt wird.

In [28]:
df_stage2 = df_norm_loaded.copy().reset_index(drop=True)

# Unique document ID
df_stage2['doc_id'] = ['doc_' + str(i).zfill(5) for i in range(len(df_stage2))]

# Readable combined text: cleaned title + cleaned content
df_stage2['text_clean_light'] = (
    df_stage2['title'].fillna('') + '. ' + df_stage2['content'].fillna('')
).str.strip('. ')

# Embedding text: final normalized lemma tokens from title + content joined
df_stage2['text_for_embedding'] = df_stage2.apply(
    lambda row: ' '.join(
        (row['title_lemma_norm'] if isinstance(row['title_lemma_norm'], list) else []) +
        (row['content_lemma_norm'] if isinstance(row['content_lemma_norm'], list) else [])
    ),
    axis=1
)

# Token count
df_stage2['token_count'] = df_stage2['text_for_embedding'].str.split().str.len()

# Reproducible 80/20 train/val split
np.random.seed(42)
split_mask = np.random.rand(len(df_stage2)) < 0.8
df_stage2['split'] = np.where(split_mask, 'train', 'val')


## 1.3 Filter, Export and Verify

Before exporting we apply three quality filters to ensure only meaningful documents reach the embedding model:

1. **Remove empty embedding texts** — rows where `text_for_embedding` is blank or whitespace-only are dropped. These would produce zero-length inputs that crash or corrupt model training. 🟨🟨🟨🟨 Marisa: Wie kommen blank rows zustande? Wir hatten in Stage 1 nach missing values gefiltert?
2. **Remove very short texts** — rows with fewer than 5 tokens are dropped. Documents this short carry almost no semantic signal and can distort word co-occurrence statistics in Word2Vec / FastText.
3. **Drop embedding duplicates** — rows whose `text_for_embedding` is identical to another row are deduplicated. Repeated documents would over-weight certain terms in the trained embeddings. 🟨🟨🟨🟨 Marisa: Ebenso hier, wir hatten duplicates doch eigentlich schon entfernt?

The index is then reset to close any gaps left by the filters, ensuring clean positional access. The final dataframe is narrowed to the 11 export columns and written to three CSV files(all UTF-8):
- `ai_media_stage2_full.csv`: The full dataset
- `ai_media_stage2_train.csv`: The dataset used for training (80% of the whole data set)
- `ai_media_stage2_val.csv`: The dataset used for validation of the trained model.  (20% of the whole data set)

🟨🟨🟨🟨 Marisa: Ich habe oben noch eine Erklärung der Datensätze beigefügt. Bitte kurz kontrollieren.

A summary is printed showing the final shape, train/val counts, and a sample of the key columns.


In [29]:
# Remove empty / very short embedding texts
df_stage2 = df_stage2[df_stage2['text_for_embedding'].str.strip() != '']
df_stage2 = df_stage2[df_stage2['token_count'] >= 5]

# Drop duplicates based on embedding text
df_stage2 = df_stage2.drop_duplicates(subset='text_for_embedding').reset_index(drop=True)

# Final column selection
export_cols = ['doc_id', 'date', 'domain', 'url', 'tags',
               'title', 'content', 'text_clean_light',
               'text_for_embedding', 'token_count', 'split']
df_export = df_stage2[export_cols]

if ENV == "colab":
    base_path = '/content/drive/My Drive/CLT/stage_2/'
else:
    base_path = 'data/stage_2/'

os.makedirs(base_path, exist_ok=True)

# Full dataset
df_export.to_csv(base_path + 'ai_media_stage2_full.csv', index=False, encoding='utf-8')

# Train / val splits — ready for downstream supervised tasks
df_export[df_export['split'] == 'train'].to_csv(base_path + 'ai_media_stage2_train.csv', index=False, encoding='utf-8')
df_export[df_export['split'] == 'val'].to_csv(base_path + 'ai_media_stage2_val.csv', index=False, encoding='utf-8')

# Summary
print("Export shape:", df_export.shape)
print("\nSplit counts:")
print(df_export['split'].value_counts())
print("\nSample (first 3 rows):")
sample = df_export[['doc_id', 'text_clean_light', 'text_for_embedding', 'token_count', 'split']].head(3).copy()
sample['text_clean_light'] = sample['text_clean_light'].str[:80] + '...'
sample['text_for_embedding'] = sample['text_for_embedding'].str[:80] + '...'
print(sample.to_string())

Export shape: (16518, 11)

Split counts:
split
train    13224
val       3294
Name: count, dtype: int64

Sample (first 3 rows):
      doc_id                                                                     text_clean_light                                                                   text_for_embedding  token_count  split
0  doc_00000  ricoh to provide customer support for agility robotics digit humanoid. the digit...  ricoh provide customer support agility robotic digit humanoid digit humanoid wor...          501  train
1  doc_00001  ai design trends to watch in 2025 from photorealism to personalization. artifici...  ai design trend watch photorealism personalization artificial intelligence ai ra...          804    val
2  doc_00002  ais generate more novel and exciting research ideas than human experts. the firs...  ais generate novel exciting research idea human expert statistically significant...          533  train


# 2 Word2Vec Embeddings

*Lead: Alla - Review: Marisa*

## 2.1 Load & Tokenize


We load the training split from `ai_media_stage2_train.csv`.

A small normalization pass is applied before training to remap known lemmatization artifacts:

| Original token | Replacement | Reason |
|---|---|---|
| `datum` | `data` | spaCy lemmatises "data" → "datum" |
| `medium` | `media` | spaCy lemmatises "media" → "medium" |
| `u200b` | *(dropped)* | Unicode zero-width space (U+200B) leaked as literal token |
| `https`, `http` | *(dropped)* | URL protocol fragments from incompletely stripped hyperlinks |
| hex tokens | *(dropped)* | UTF-8 mojibake artifacts (`x9d`, `x80`, `x9cthe`, etc.) |

Counts before and after are printed so any additional odd tokens can be caught and added to `TOKEN_MAP`.

In [30]:
if ENV == "colab":
    train_path = '/content/drive/My Drive/CLT/stage_2/ai_media_stage2_train.csv'
else:
    train_path = 'data/stage_2/ai_media_stage2_train.csv'

df_train = pd.read_csv(train_path, encoding='utf-8')

# Split space-separated token strings into lists — required format for gensim
sentences = [text.split() for text in df_train['text_for_embedding'].dropna()]

# --- Token normalization ---
# TOKEN_MAP: explicit remappings (old → new, empty string = drop).
# HEX_PATTERN: drops any token starting with a hex escape sequence (x9d, x9cthe, etc.)
TOKEN_MAP = {
    'datum' : 'data',    # spaCy lemmatises "data" → "datum"
    'medium': 'media',   # spaCy lemmatises "media" → "medium"
    'u200b' : '',        # Unicode zero-width space leaked as literal token — drop
    'https' : '',        # URL protocol fragment — drop
    'http'  : '',        # URL protocol fragment — drop
}
HEX_PATTERN = re.compile(r'^x[0-9a-f]{2}')   # matches x9d, x80, x9cthe, etc.

def _keep(tok: str) -> str:
    """Return the normalised token, or '' to signal drop."""
    tok = TOKEN_MAP.get(tok, tok)          # apply explicit remapping first
    if HEX_PATTERN.match(tok):             # then drop hex artifacts
        return ''
    return tok

# Check counts before fix
from collections import Counter
flat_before   = [t for s in sentences for t in s]
flat_before   = [t for s in sentences for t in s]
counts_before = Counter(flat_before)
hex_before    = {t: c for t, c in counts_before.items() if HEX_PATTERN.match(t)}
print("=== Token counts BEFORE normalization ===")
for tok in TOKEN_MAP:
    print(f"  {tok:<12} : {counts_before.get(tok, 0):,}")
print(f"  hex artifacts: {sum(hex_before.values()):,} occurrences across {len(hex_before):,} unique tokens")

# Apply normalization
sentences = [
    [out for t in s for out in [_keep(t)] if out]
    for s in sentences
]

# Check counts after fix
flat_after   = [t for s in sentences for t in s]
counts_after = Counter(flat_after)
hex_after    = {t: c for t, c in counts_after.items() if HEX_PATTERN.match(t)}
print("\n=== Token counts AFTER normalization ===")
for orig, repl in TOKEN_MAP.items():
    print(f"  {orig:<12} → {repl:<12} : {counts_after.get(repl, 0):,}")
print(f"  hex artifacts remaining: {sum(hex_after.values()):,}")

print(f"\nDocuments  : {len(sentences):,}")
print(f"Total tokens: {sum(len(s) for s in sentences):,}")
print(f"Sample     : {sentences[0][:12]} ...")

=== Token counts BEFORE normalization ===
  datum        : 74,577
  medium       : 10,143
  u200b        : 630
  https        : 3,023
  http         : 351
  hex artifacts: 40,403 occurrences across 2,461 unique tokens

=== Token counts AFTER normalization ===
  datum        → data         : 91,666
  medium       → media        : 10,793
  u200b        →              : 0
  https        →              : 0
  http         →              : 0
  hex artifacts remaining: 0

Documents  : 13,224
Total tokens: 11,296,257
Sample     : ['ricoh', 'provide', 'customer', 'support', 'agility', 'robotic', 'digit', 'humanoid', 'digit', 'humanoid', 'work', 'distribution'] ...


🟨🟨🟨🟨 Marisa: Erkläre hier bitte, was du als nächstes machst und warum.

In [31]:
flat_tokens = Counter(t for s in sentences for t in s)
TOP_FREQ    = 50   # show this many of the most frequent suspicious tokens per category

suspicious = {
    'hex_artifacts'  : {},   # x80, xc2, etc.
    'non_ascii'      : {},   # contains chars outside basic ASCII
    'numeric_only'   : {},   # pure numbers like "2023", "100"
    'single_char'    : {},   # single character tokens
    'punctuation'    : {},   # contains non-alphanumeric characters
}

for tok, cnt in flat_tokens.items():
    if re.fullmatch(r'x[0-9a-f]{2}', tok):
        suspicious['hex_artifacts'][tok] = cnt
    elif not tok.isascii():
        suspicious['non_ascii'][tok] = cnt
    elif tok.isnumeric():
        suspicious['numeric_only'][tok] = cnt
    elif len(tok) == 1:
        suspicious['single_char'][tok] = cnt
    elif not tok.isalpha():
        suspicious['punctuation'][tok] = cnt

for category, tokens in suspicious.items():
    top = sorted(tokens.items(), key=lambda x: x[1], reverse=True)[:TOP_FREQ]
    if top:
        print(f"\n{'='*50}")
        print(f"  {category.upper()}  ({len(tokens):,} unique tokens)")
        print(f"{'='*50}")
        for tok, cnt in top:
            print(f"  {tok:<20} : {cnt:,}")
    else:
        print(f"\n  {category}: ✓ none found")



  hex_artifacts: ✓ none found

  non_ascii: ✓ none found

  numeric_only: ✓ none found

  single_char: ✓ none found

  PUNCTUATION  (8,907 unique tokens)
  3d                   : 1,896
  b2b                  : 1,122
  r1                   : 1,107
  4o                   : 748
  q1                   : 646
  o1                   : 635
  a16z                 : 572
  4th                  : 553
  q4                   : 539
  15th                 : 482
  3rd                  : 457
  q2                   : 452
  co2                  : 440
  q3                   : 425
  o3                   : 415
  sc24                 : 378
  j916                 : 369
  yolov10              : 367
  h100                 : 352
  20th                 : 349
  21st                 : 322
  v3                   : 316
  1b                   : 309
  30th                 : 292
  g7                   : 283
  1st                  : 276
  4k                   : 262
  a2a                  : 260
  eur1                 : 25

## 2.1.1 Suspicious Token Scan

**Expected tokens in the `punctuation` category** — alphanumeric tokens containing digits are not artifacts; they are legitimate corpus vocabulary:

| Token(s) | Type | Examples |
|---|---|---|
| `3d`, `2d`, `4k` | Dimensional / display spec | 3D rendering, 4K video |
| `b2b`, `b2c` | Business model abbreviation | business-to-business, business-to-consumer |
| `r1`, `o1`, `o3`, `4o` | AI model names | DeepSeek R1, OpenAI o1/o3, GPT-4o |
| `q1` – `q4` | Fiscal quarters | Q1 earnings, Q4 results |
| `8b`, `70b`, `1b`, `3b` | Model parameter scale | 8-billion-parameter model |
| `v1`, `v2`, `v3` | Version numbers | model version, API version |
| `h100`, `gb200` | GPU / hardware names | NVIDIA H100, GB200 |
| `10x`, `5x` | Growth multipliers | 10× revenue growth |
| `a16z` | VC fund shorthand | Andreessen Horowitz |
| `web3`, `co2`, `g7` | Domain terms | Web3 ecosystem, CO₂ emissions, G7 summit |
| ordinals (`1st`…`30th`) | Date ordinals | published on the 15th |

These should all be **kept** — they carry real semantic signal for the AI-media domain.


## 2.2 Train Word2Vec



We train a **Skip-gram Word2Vec** model on the `text_for_embedding` column of the training split.
The input tokens are the pre-processed lemma sequences produced in Stage 1 (stop words removed, normalised).

**Role in the broader pipeline:**  
Word2Vec operates on individual lemma tokens, not BPE subwords, so its vectors do not plug directly into a TinyLlama-style transformer. 🟨🟨🟨🟨 Marisa: Was heisst BPE? Schreibe es in Klammern aus.

They serve as:
- a **domain vocabulary sanity check** — nearest-neighbour probes reveal whether the corpus signal is coherent;
- **embedding initialisation** for lightweight classification or retrieval heads built on top of the LLM;
- **input features** for non-neural baselines (TF-IDF + W2V centroid classifiers, etc.).

**Hyperparameters chosen:**

| Parameter | Value | Rationale |
|---|---|---|
| `vector_size` | 200 | Good balance for ~16 k documents |
| `window` | 5 | Standard context window |
| `min_count` | 5 | Drop tokens appearing fewer than 5 times |
| `sg` | 1 | Skip-gram; better for rare domain terms |
| `epochs` | 10 | Sufficient for convergence on this corpus size |


Training uses all available CPU cores.

In [32]:
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=200,   # embedding dimensions
    window=5,          # context window
    min_count=5,       # ignore tokens appearing fewer than 5 times
    sg=1,              # 1 = Skip-gram, 0 = CBOW
    workers=multiprocessing.cpu_count(),
    epochs=10,
    seed=42,
)

print(f"Vocabulary size : {len(w2v_model.wv):,} tokens")
print(f"Vector dimensions: {w2v_model.wv.vector_size}")


Vocabulary size : 42,155 tokens
Vector dimensions: 200


## 2.3 Save Model



Two artefacts are saved to `data/stage_2/embeddings/`:

- **`w2v_ai_media.model`** — full gensim model (can be reloaded for further training or inference). 🟨🟨🟨🟨 Marisa: Was heisst gensim?
- **`w2v_ai_media_vectors.txt`** — plain-text word vectors in `word2vec` format, compatible with external tools (FastText, Magnitude, `load_facebook_vectors`, etc.).

In [33]:
if ENV == "colab":
    model_dir = '/content/drive/My Drive/CLT/stage_2/embeddings/'
else:
    model_dir = 'data/stage_2/embeddings/'

os.makedirs(model_dir, exist_ok=True)

# Full gensim model (reloadable)
w2v_model.save(model_dir + 'w2v_ai_media.model')

# Plain-text vectors (word2vec format — compatible with external tools)
w2v_model.wv.save_word2vec_format(model_dir + 'w2v_ai_media_vectors.txt', binary=False)

print("Saved to:", model_dir)


Saved to: /content/drive/My Drive/CLT/stage_2/embeddings/


## 2.4 Evaluate



Nearest-neighbour probes confirm the model has learned domain-coherent representations.
We test a small set of AI-media terms `probe_words`.
We also print the top-20 tokens by frequency to quickly spot any noise that survived the earlier cleaning steps.

In [34]:
wv = w2v_model.wv

# --- Nearest-neighbour probes ---
probe_words = ['ai', 'model', 'work', 'bias', 'security']

print("=== Nearest-neighbour probes (top 5) ===")
for word in probe_words:
    if word in wv:
        neighbours = wv.most_similar(word, topn=5)
        neighbours_str = ', '.join(f"{w} ({s:.2f})" for w, s in neighbours)
        print(f"  {word:<12} → {neighbours_str}")
    else:
        print(f"  {word:<12} → not in vocabulary")

# --- Top-20 tokens by frequency ---
print("\n=== Top-20 tokens by frequency ===")
vocab_counts = sorted(wv.key_to_index.keys(),
                      key=lambda w: w2v_model.wv.get_vecattr(w, 'count'),
                      reverse=True)[:20]
for i, w in enumerate(vocab_counts, 1):
    count = w2v_model.wv.get_vecattr(w, 'count')
    print(f"  {i:>2}. {w:<20} {count:,}")


=== Nearest-neighbour probes (top 5) ===
  ai           → genai (0.67), generative (0.60), mmt (0.56), ragop (0.55), illuma (0.55)
  model        → modelsa (0.71), autorater (0.68), jamba (0.67), pixtral (0.66), grm (0.65)
  work         → collaborate (0.57), talente (0.55), costigan (0.55), teramind (0.54), worka (0.54)
  bias         → biased (0.66), fairness (0.65), perpetuate (0.63), discrimination (0.61), perpetuation (0.60)
  security     → cybersecurity (0.70), sentra (0.54), posture (0.54), getvisibility (0.54), carbonite (0.53)

=== Top-20 tokens by frequency ===
   1. ai                   163,854
   2. data                 91,666
   3. use                  78,932
   4. model                72,459
   5. company              65,933
   6. new                  47,417
   7. time                 45,220
   8. like                 44,973
   9. technology           43,465
  10. system               41,884
  11. year                 39,739
  12. business             37,399
  13. work  

**Embedding evaluation**

The nearest-neighbour analysis indicates that the embeddings capture meaningful semantic relationships in the dataset. For instance, “bias” is closely associated with fairness, biased, and intersectionality, reflecting discussions around AI ethics and fairness. Similarly, “security” is linked to cybersecurity, protection, and cyber, suggesting a coherent cybersecurity cluster. The token “model” is also connected to technical AI-related terms such as mllm and xlnet, indicating that the model groups machine learning concepts effectively. In contrast, more general words like “work” show weaker semantic coherence, which is expected due to their broad contextual usage.

**Dataset structure**

The most frequent tokens show that the dataset primarily discusses AI technologies and industry adoption. Terms such as model, data, technology, and company dominate the vocabulary, confirming the dataset’s focus on technological developments and business applications of AI

# 3 Cluster Visualisation

*Lead: Alla - Review: Marisa*

We project the Word2Vec **vocabulary** into 2D using **t-SNE**, then assign each word to a thematic cluster using **K-Means**.
Each point in the interactive plot is one vocabulary token. Points that are close together share similar contexts in the corpus — hovering reveals the word and its frequency.

**Pipeline:**
1. Take the top `TOP_N` most-frequent words (keeps the plot readable and focuses on signal-rich terms).
2. L2-normalise the vectors (equivalent to cosine similarity for K-Means).
3. Reduce to 2D with t-SNE (`perplexity=40`, `n_iter=1000`).
4. Cluster with K-Means (`N_CLUSTERS` groups).
5. Render as an interactive Plotly scatter — point size scales with token frequency.


## 3.1 Configure & Extract Vectors



Adjust `TOP_N` and `N_CLUSTERS` to trade off detail vs. readability.
A good starting point is 2 000–4 000 words and 6–12 clusters for an AI-media corpus of this size.

🟨🟨🟨🟨 Marisa: Was ist im Code die Basis von den Theme anchors? Wo kommen die Begriffe her, aus Stage 1 nicht, oder? Bitte erklären.

In [35]:
TOP_N      = 3000   # most-frequent words to include in the plot
N_CLUSTERS = 12     # number of thematic clusters

# Anchor words for each theme — the cluster whose vocabulary best matches
# a set of anchors gets that name automatically (robust to ID shuffling across runs).
THEME_ANCHORS = {
    'AI Models & Data'      : ['ai', 'model', 'data', 'language', 'llm', 'train', 'neural', 'transformer', 'vector', 'embed'],
    'Industry & Business'   : ['company', 'business', 'market', 'customer', 'product', 'revenue', 'startup', 'enterprise', 'invest', 'technology'],
    'Policy & Governance'   : ['policy', 'regulation', 'governance', 'government', 'law', 'right', 'global', 'rule', 'compliance', 'share', 'value'],
    'Research & Society'    : ['research', 'study', 'human', 'social', 'result', 'find', 'scientist', 'university', 'paper', 'new', 'time', 'year'],
    'Systems & Tools'       : ['system', 'software', 'platform', 'deploy', 'infrastructure', 'automate', 'integrate', 'workflow', 'use', 'help'],
    'Media & Information'   : ['content', 'information', 'access', 'news', 'media', 'publish', 'journalist', 'article', 'report'],
    'General Narrative'     : ['say', 'call', 'note', 'describe', 'suggest', 'believe', 'accord', 'add', 'explain', 'like', 'need', 'lead'],
    'Ethics & Safety'       : ['bias', 'safety', 'harm', 'risk', 'ethical', 'responsible', 'privacy', 'fairness', 'trust', 'accountability'],
    'Investment & Finance'  : ['fund', 'invest', 'capital', 'million', 'billion', 'valuation', 'raise', 'acquisition', 'deal', 'vc'],
    'Innovation & Development': ['build', 'develop', 'drive', 'change', 'improve', 'design', 'create', 'launch', 'engineer', 'solution'],
    'Education & Workforce' : ['student', 'education', 'skill', 'job', 'worker', 'learn', 'school', 'career', 'employment', 'talent'],
    'Healthcare & Science'  : ['health', 'medical', 'drug', 'patient', 'clinical', 'diagnosis', 'hospital', 'biology', 'disease', 'science'],
}

# Sort vocabulary by frequency descending, take top N
words_sorted = sorted(
    wv.key_to_index.keys(),
    key=lambda w: wv.get_vecattr(w, 'count'),
    reverse=True
)[:TOP_N]

vectors = np.array([wv[w] for w in words_sorted])
freqs   = np.array([wv.get_vecattr(w, 'count') for w in words_sorted])



print(f"Words selected : {len(words_sorted):,}")
print(f"Vector matrix  : {vectors.shape}")
print(f"Frequency range: {freqs.min():,} – {freqs.max():,}")

Words selected : 3,000
Vector matrix  : (3000, 200)
Frequency range: 498 – 163,854


## 3.2 Optimal K — Elbow & Silhouette

Before fixing `N_CLUSTERS` we sweep K = 2 … 12 on the L2-normalised vectors and plot two diagnostics:

- **Elbow (inertia)** — within-cluster sum of squares; look for the point where the curve bends and further gains shrink.
- **Silhouette score** — mean similarity of each word to its own cluster vs. the next-best cluster; ranges 0–1, higher is better; pick the peak.

Both charts use the same high-dimensional normalised vectors (not the 2D t-SNE projection) so the K choice is grounded in the actual embedding geometry.


In [36]:
K_RANGE = range(2, 13)

vectors_norm_k = normalize(vectors)   # L2-normalise for cosine-like K-Means

inertias    = []
silhouettes = []

for k in K_RANGE:
    km_k = KMeans(n_clusters=k, random_state=42, n_init='auto')
    lbl  = km_k.fit_predict(vectors_norm_k)
    inertias.append(km_k.inertia_)
    silhouettes.append(silhouette_score(vectors_norm_k, lbl, sample_size=1500, random_state=42))
    print(f"  k={k:2d}  inertia={km_k.inertia_:,.0f}  silhouette={silhouettes[-1]:.4f}")

# --- Plot both diagnostics side-by-side ---
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_k = make_subplots(rows=1, cols=2,
                      subplot_titles=('Elbow — Inertia (lower is better)',
                                      'Silhouette Score (higher is better)'))

fig_k.add_trace(go.Scatter(x=list(K_RANGE), y=inertias,
                            mode='lines+markers', name='Inertia',
                            line=dict(color='steelblue', width=2),
                            marker=dict(size=7)), row=1, col=1)

fig_k.add_trace(go.Scatter(x=list(K_RANGE), y=silhouettes,
                            mode='lines+markers', name='Silhouette',
                            line=dict(color='tomato', width=2),
                            marker=dict(size=7)), row=1, col=2)

# Mark the chosen N_CLUSTERS on both subplots
for col in [1, 2]:
    fig_k.add_vline(x=N_CLUSTERS, line_dash='dash', line_color='grey',
                    annotation_text=f'N_CLUSTERS={N_CLUSTERS}',
                    annotation_position='top right', row=1, col=col)

fig_k.update_xaxes(title_text='K', dtick=1)
fig_k.update_layout(title_text='Optimal K selection for K-Means clustering',
                    width=1000, height=420, showlegend=False)
fig_k.show()


  k= 2  inertia=2,491  silhouette=0.0184
  k= 3  inertia=2,461  silhouette=0.0146
  k= 4  inertia=2,436  silhouette=0.0159
  k= 5  inertia=2,417  silhouette=0.0149
  k= 6  inertia=2,398  silhouette=0.0169
  k= 7  inertia=2,381  silhouette=0.0173
  k= 8  inertia=2,368  silhouette=0.0183
  k= 9  inertia=2,353  silhouette=0.0190
  k=10  inertia=2,343  silhouette=0.0188
  k=11  inertia=2,332  silhouette=0.0195
  k=12  inertia=2,319  silhouette=0.0199


**K-Means Cluster Evaluation**

To determine the appropriate number of clusters, we evaluated K-Means performance for K = 2–12 using both inertia (elbow method) and silhouette score.

The inertia values decrease steadily as K increases (from 2491 at K=2 to 2320 at K=12), which is expected since adding more clusters reduces within-cluster variance. However, the decrease is gradual and does not show a clear elbow point, suggesting that the dataset does not have strongly separated cluster boundaries.

The silhouette scores are relatively low across all K values (≈0.016–0.020), indicating that the clusters are only weakly separated in the embedding space. This is common in high-dimensional semantic embeddings where topics overlap. The highest silhouette score occurs at K=11 (0.0201), with K=12 (0.0198) showing a very similar value.

Based on these diagnostics, 12 clusters were selected. Although the improvement is modest, this value provides slightly better cluster separation while allowing for more granular thematic grouping of AI-related topics in the dataset.

## 3.3 t-SNE Reduction & K-Means Clustering

Vectors are L2-normalised before t-SNE so distances approximate cosine similarity.
t-SNE takes ~30–60 s on 3 000 points with `n_jobs=1` (single-threaded to avoid Cython issues).

K-Means is applied **on the 2D t-SNE coordinates** (not the original 200-dim vectors).
This ensures every cluster forms a spatially contiguous region in the plot — no colour will appear in two separate corners.


In [37]:
# L2-normalise (cosine similarity)
vectors_norm = normalize(vectors)

# t-SNE: reduce 200-dim → 2-dim
print("Running t-SNE …")
tsne   = TSNE(n_components=2, perplexity=40, max_iter=1000, random_state=42, n_jobs=1)
coords = tsne.fit_transform(vectors_norm)

# K-Means on 2D t-SNE coords — clusters will be spatially coherent in the plot
km     = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init='auto')
labels = km.fit_predict(coords)

print("Done.\nCluster sizes:")
unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
    mask      = labels == u
    top_words = [words_sorted[i] for i in np.where(mask)[0][:5]]
    print(f"  Cluster {u} ({c:>4} words) — sample: {', '.join(top_words)}")


Running t-SNE …
Done.
Cluster sizes:
  Cluster 0 ( 213 words) — sample: energy, operation, center, number, space
  Cluster 1 ( 301 words) — sample: use, like, need, help, include
  Cluster 2 ( 261 words) — sample: world, global, policy, state, china
  Cluster 3 ( 199 words) — sample: system, tool, platform, process, solution
  Cluster 4 ( 269 words) — sample: new, work, lead, way, change
  Cluster 5 ( 240 words) — sample: ai, data, model, large, build
  Cluster 6 ( 285 words) — sample: time, people, good, come, think
  Cluster 7 ( 271 words) — sample: high, cloud, performance, quantum, improve
  Cluster 8 ( 298 words) — sample: security, risk, cost, increase, long
  Cluster 9 ( 239 words) — sample: create, user, human, design, experience
  Cluster 10 ( 206 words) — sample: year, research, team, read, day
  Cluster 11 ( 218 words) — sample: company, technology, business, customer, service


**Cluster Interpretation**

The clustering reveals 12 thematic dimensions of AI discourse in the dataset. These clusters can be grouped into several broader categories:

**1. Technical AI Development** 🟨🟨🟨🟨 Marisa: Gibt es einen grund, dass die Cluster nicht von niedrigster zur höchsten Zahl sortiert sind? Wenn nein, würde ich sie so sortieren. --> ACHTUNG, werden bei jeder Ausführung anders.

`Cluster 8` – AI Models & Data: data, model, large, language

`Cluster 7` – Systems & Performance: performance, quantum, network

`Cluster 2` – AI Systems & Tools: ai, system, use, help

These clusters represent the technical foundations of AI, including models, infrastructure, and system applications.

**2. Business & Industry**

`Cluster 11 `– Business & Technology: company, technology, business

`Cluster 10` – Industry & Operations: industry, product, energy

These clusters capture commercial adoption and sector-specific deployment of AI technologies.

**3. Governance & Risk**

`Cluster 0` – Ethics & Safety: security, risk, policy

`Cluster 3` – Policy & Geopolitics: global, state, china

They reflect discussions about AI regulation, security, and geopolitical competition.

**4. Research & Social Context**

`Cluster 9 `– Research & Society: research, team, study

`Cluster 1` – Education & Workforce / Healthcare: people, patient

These clusters represent academic research and AI’s societal impact, including healthcare and workforce themes.

**5. Media & Narrative Language**

`Cluster 5` – Media & Human Intelligence: human, content, intelligence

`Cluster 6` – Innovation & Development: work, change

`Cluster 4` – General Narrative: like, need, result

These clusters capture editorial, narrative, and evaluative language common in media discussions.

Overall, the clusters show that the dataset covers technical AI development, business adoption, governance, research, and societal implications, reflecting the multidimensional nature of AI discourse.

🟨🟨🟨🟨 Marisa: Das sind ja nicht dieselben Themes, die wir oben definiert haben bei den Anchors. Mir fehlt hier noch die Verbindung zwischen den Anchor Themes und diessen Clustern.

## 3.4 Interactive Plot

Each point is one vocabulary word.
**Size** scales with token frequency — larger dots are the most common terms in the corpus.
**Colour** represents the K-Means cluster.
Hover any point to see the word and its raw frequency count.


In [38]:

# --- Build top-words lookup per cluster ---
cluster_top_words = {}
cluster_word_sets = {}
for cid in sorted(np.unique(labels)):
    idx_sorted = sorted(np.where(labels == cid)[0], key=lambda i: freqs[i], reverse=True)
    cluster_top_words[cid] = [words_sorted[i] for i in idx_sorted[:3]]
    cluster_word_sets[cid]  = set(words_sorted[i] for i in np.where(labels == cid)[0])

# --- Auto-assign theme names via anchor-word matching ---
# Each cluster gets the theme whose anchors overlap most with its vocabulary.
# Each theme is assigned at most once (greedy best-match).
assigned_themes = {}
used_themes     = set()
scores = {
    cid: {
        # Normalize by anchor count so small clusters aren't penalised vs. large ones
        theme: len(cluster_word_sets[cid] & set(anchors)) / len(anchors)
        for theme, anchors in THEME_ANCHORS.items()
    }
    for cid in sorted(np.unique(labels))
}
all_pairs = sorted(
    [(cid, theme) for cid in scores for theme in scores[cid]],
    key=lambda p: scores[p[0]][p[1]], reverse=True
)
for cid, theme in all_pairs:
    if cid not in assigned_themes and theme not in used_themes:
        assigned_themes[cid] = theme
        used_themes.add(theme)
for cid in sorted(np.unique(labels)):
    if cid not in assigned_themes:
        assigned_themes[cid] = ' / '.join(cluster_top_words[cid])

print("Auto-assigned cluster names:")
for cid, name in sorted(assigned_themes.items()):
    print(f"  Cluster {cid}: {name}  (top words: {', '.join(cluster_top_words[cid])})")

def _cluster_label(cid):
    return assigned_themes[cid]

# Centroid coordinates in t-SNE space
centroids = {
    cid: (coords[labels == cid, 0].mean(), coords[labels == cid, 1].mean())
    for cid in sorted(np.unique(labels))
}

df_viz = pd.DataFrame({
    'x'      : coords[:, 0],
    'y'      : coords[:, 1],
    'word'   : words_sorted,
    'cluster': [_cluster_label(l) for l in labels],
    'freq'   : freqs,
})

fig = px.scatter(
    df_viz,
    x='x', y='y',
    color='cluster',
    hover_name='word',
    hover_data={'freq': ':,', 'cluster': True, 'x': False, 'y': False},
    size='freq',
    size_max=25,
    opacity=0.70,
    title=f't-SNE of Word2Vec vocabulary — {N_CLUSTERS} K-Means clusters  (top {TOP_N:,} words)',
    width=1200,
    height=840,
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.update_traces(marker=dict(line=dict(width=0)))

# Centroid label boxes — cluster name
for cid, (cx, cy) in centroids.items():
    fig.add_annotation(
        x=cx, y=cy,
        text=f'<b>{_cluster_label(cid)}</b>',
        showarrow=False,
        font=dict(size=12, color='#111'),
        bgcolor='rgba(255,255,255,0.82)',
        bordercolor='#999',
        borderwidth=1,
        borderpad=5,
    )

fig.update_layout(
    legend_title_text='Cluster',
    legend=dict(font=dict(size=11)),
    xaxis=dict(showticklabels=False, title='', showgrid=False, zeroline=False),
    yaxis=dict(showticklabels=False, title='', showgrid=False, zeroline=False),
)
fig.show()



Auto-assigned cluster names:
  Cluster 0: Ethics & Safety  (top words: energy, operation, center)
  Cluster 1: Research & Society  (top words: use, like, need)
  Cluster 2: Policy & Governance  (top words: world, global, policy)
  Cluster 3: Systems & Tools  (top words: system, tool, platform)
  Cluster 4: General Narrative  (top words: new, work, lead)
  Cluster 5: AI Models & Data  (top words: ai, data, model)
  Cluster 6: Media & Information  (top words: time, people, good)
  Cluster 7: Investment & Finance  (top words: high, cloud, performance)
  Cluster 8: Healthcare & Science  (top words: security, risk, cost)
  Cluster 9: Innovation & Development  (top words: create, user, human)
  Cluster 10: Education & Workforce  (top words: year, research, team)
  Cluster 11: Industry & Business  (top words: company, technology, business)


**Interpretation of the t-SNE Visualization**

🟨🟨🟨🟨 Update Marisa: Positionen sind bei jeder Ausführung anders. Deshalb sind meine Kommentare nicht mehr ganz akkurat. Überprüfe, ob allgemeine Aussagen wie X, Y und Z sind nah beieinander immer gelten (müsste ja).

The t-SNE projection illustrates the semantic organisation of the Word2Vec vocabulary, where words with similar contextual usage appear close to each other. The resulting clusters reflect several thematic areas within the AI discourse dataset.

**Spatial organisation**

The plot shows a thematic structure with partially overlapping regions, which is typical for media discourse where topics are interconnected.

**Technical AI topics** are mainly located on the right side of the plot. 🟨🟨🟨🟨 Marisa: Wenn ich diesen Satz lese, schaue ich als erstes zu Healthcare & Science, was ja eher "domain-usage" o.ä. wäre?


*   **AI Models & Data and Systems & Tools** appear close together
reflecting strong semantic relationships between model development, infrastructure, and AI tools.
*   **Innovation & Development** is nearby, capturing language related to building and improving AI systems.

**Business and economic** themes are positioned in the central-left region.


*   **Industry & Business and Investment & Finance clusters** indicate discussions about market adoption and financial aspects of AI technologies.


**Governance and societal topics** appear more toward the left and upper areas.



*   **Policy & Governance, Ethics & Safety, and Research & Society** are located relatively close to each other, highlighting the connection between regulation, ethical considerations, and research discourse.




**Social and sectoral themes** appear in the lower region, including Education & Workforce, Healthcare & Science, and Media & Information, which represent societal domains influenced by AI. 🟨🟨🟨🟨 Marisa: Healthcare und Science ist definitiv in der oberen Hälfte.

**The General Narrative** cluster is located near the center of the plot, which is expected. This cluster contains common evaluative and connective words that appear across many contexts, so it naturally overlaps with multiple thematic regions and forms a central hub in the semantic space.

**Limitations of the 2D projection**

It is important to note that this visualization represents a 2-dimensional projection of a high-dimensional embedding space. Consequently:

- Distances between clusters may be distorted due to dimensionality reduction.

- t-SNE preserves local neighbourhood relationships better than global structure.

- Clusters that appear close in the plot may not be equally close in the original embedding space.

Therefore, the visualization should be interpreted as an approximate representation of semantic relationships, useful for identifying thematic patterns but not for precise distance comparisons.

# 4 Finetuning

*Lead: Marisa - Review: Alla*

## 4.1 Model Selection

In this section we select the base model for our AI trend analysis. We choose `TinyLlama 1.1B`. Large Language Models normally require massive computational resources which makes them difficult to run on standard machines. TinyLlama provides an excellent balance. It is small enough to run on environments with limited memory like Google Colab but capable enough to understand complex text patterns. We load the model in 16 bit precision using the torch float16 data type. This halves the memory requirement compared to standard 32 bit precision without significantly losing accuracy.

In [42]:
# Define the model identifier
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load the model with memory optimisation
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning:


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.



config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


## 4.2 Tokenization and Data Formatting

Before the model can learn from our AI Media Dataset we must convert the text into numbers. The Tokeniser breaks down our cleaned texts into tokens. We also set a padding token. Neural networks process data in batches of equal size. Padding ensures that shorter texts are filled up with empty tokens to match the length of the longest text in a given batch.

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Set padding token to avoid errors during batch processing
tokenizer.pad_token = tokenizer.eos_token

## 4.3 Parameter Efficient Fine Tuning and LoRA

Standard fine tuning updates all weights inside a model. For a 1.1 billion parameter model this is incredibly memory intensive. Therefore we use Parameter Efficient Fine Tuning (PEFT) combined with Low Rank Adaptation (LoRA). LoRA freezes the original model weights and only trains a very small set of new parameters inserted into specific layers. This reduces the trainable parameters to less than 1 percent making the training much faster and perfectly suited for our hardware constraints.

In [ ]:
# Configure LoRA settings
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to the base model
model = get_peft_model(model, lora_config)

# Display the number of trainable parameters
model.print_trainable_parameters()

## 4.4 Training Configuration

We configure the training arguments to guide the learning process. We set the learning rate to 0.0002 which dictates how big the update steps are during training. A batch size of 4 means the model looks at four examples at a time before updating its knowledge. We limit the training to 1'000 steps. This provides a good middle ground to ensure a manageable execution time while still allowing the model to adapt to the AI discourse present in our dataset.

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./ai_trend_model",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=1000,
    optim="adamw_torch",
    save_strategy="epoch",
    fp16=True
)

## 4.5 Execution and Model Saving

Finally we initialise the Supervised Fine Tuning Trainer. This tool simplifies the training loop by automatically applying our formatting and LoRA layers. Once the 1'000 steps are completed we save the new weights. These weights act as a small patch that we can later load on top of the original TinyLlama model to give it our domain specific knowledge about AI trends.

In [ ]:
# Initialise the trainer
# Note train_dataset must be your prepared dataset from Stage 1
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=512,
    args=training_args
)

# Start the training process
trainer.train()

# Save the fine tuned LoRA adapter
trainer.save_model("./ai_trend_lora_adapter")